# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aryan-0018/FlyRank-ML-W1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
!git clone https://github.com/aryan-0018/FlyRank-ML-W1.git

fatal: destination path 'FlyRank-ML-W1' already exists and is not an empty directory.


In [17]:
from pathlib import Path

guide_path = Path("FlyRank-ML-W1/GUIDE.md")

print("GUIDE exists:", guide_path.exists())


GUIDE exists: True


In [18]:
print(Path("FlyRank-ML-W1/GUIDE.md").read_text())

# GUIDE — how this repo works

Five minutes here saves you hours later. This is the operating manual: what every file is,
what you may edit, how the pieces connect, and where your own work goes.

## 1. The map

| Path | What it is | Your relationship to it |
|---|---|---|
| `README.md` | Front door: quickstart + safety summary | Read once |
| `GUIDE.md` | This file | Read once, revisit when unsure |
| `SETUP.md` | GitHub + Colab (Week 1) and Hugging Face access (Week 3) — with the silent pitfalls | Follow it at those two moments |
| `DATA_USE.md` | The data rules you agree to by working here | **Read before touching the data** |
| `LICENSE` | MIT — covers the **code** only; the data is governed by `DATA_USE.md` | Reference |
| `data/raw/content_refresh_anonymized.csv` | The one dataset that ships here: 30,000 pseudonymized pages × 44 columns | **Read-only.** Never add files under `data/` |
| `data/processed/` | Intermediate files the pipeline writes (created on first run; gitignored) |

In [19]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [20]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [21]:
import os

os.environ["HF_TOKEN"] = HF_TOKEN

print("Hugging Face access token configured for this session.")

Hugging Face access token configured for this session.


In [22]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE SECRET (
    TYPE huggingface,
    TOKEN ?
)
""", [HF_TOKEN])

print("DuckDB connected successfully.")
print("Hugging Face access configured in DuckDB.")

DuckDB connected successfully.
Hugging Face access configured in DuckDB.


In [23]:
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
TABLE_PATH = f"{WAREHOUSE}/fact_content_daily_performance/**/*.parquet"

print("Warehouse path configured.")

Warehouse path configured.


In [24]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{TABLE_PATH}',
    hive_partitioning=true
)
LIMIT 1
""")

display(schema.df())

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [25]:
date_check = con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS row_count
FROM read_parquet(
    '{TABLE_PATH}',
    hive_partitioning=true
)
""")

display(date_check.df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,row_count
0,2025-01-27,2026-06-30,78835655


In [26]:
from pathlib import Path

print(Path("FlyRank-ML-W1/DATA_USE.md").read_text())

# Data Use & Public-Safety Rules

The one dataset that ships with this repo is:

```text
data/raw/content_refresh_anonymized.csv
```

It is a small, **anonymized** slice of FlyRank content-performance data — one row per
pseudonymized content item, with observed search/engagement metrics, content metadata,
age/freshness fields, and derived comparison windows.

## What has already been removed

The starter export contains **no**:

- client names
- domains
- URLs
- page titles
- keywords or raw search queries
- product-rule flags used as composite scores you should trust blindly

Only hashed `content_id` / `client_id` labels plus numeric and categorical metrics remain.

Rate columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`) are
percentages on a 0–100 scale: `ctr = 0.76` means 0.76%, not 76%.

The hashed IDs are pseudonyms derived from FlyRank-internal database identifiers. They
contain no public information, but treat them as pseudonymous, not anonymous: use

In [27]:
lane_guide = Path(
    "FlyRank-ML-W1/docs/ml-intern-dataset-and-lane-guide.md"
)

print(lane_guide.read_text())

# FlyRank ML Internship - Your Dataset and Lane Guide

Status: your guide for the Applied Search Intelligence track.

Read this together with:

- `docs/ml-core-foundation-framework.md` (ships in this repo — a deep reference, not week-one reading)
- `docs/intern-free-tooling-guide.md` (ships in this repo)
- the week-by-week curriculum on your portal board
- the data dictionary and manifest that ship inside the dataset release on Hugging Face

> This starter repo ships only the small anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`).
> The much larger warehouse release lives on Hugging Face at
> [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse).
> It is gated, which means you click "request access" and accept the data-use terms first —
> approval is instant, so you can get it yourself whenever your lane needs it.

This guide shows you how to use the data without turning the internship into a fill-in-the-blank exercise. The

In [28]:
nb03 = Path(
    "FlyRank-ML-W1/notebooks/03_working_with_the_full_release.ipynb"
)

print("Notebook 03 exists:", nb03.exists())

Notebook 03 exists: True


## 1. Question

*The research question and the decision it supports.*

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
